# Does a Discount Actually Move More Units?

The store's discount folklore has two parts: cut the price, and people buy a
lot more, so revenue ends up higher regardless of the cut. This notebook tests
only the first half of that: whether putting a discount on an order line
changes how many units go out on that line. Whether the lift, if there is one,
is worth what the discount gives away is a profitability question, and it
belongs elsewhere in this project, not here.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = next(p for p in Path.cwd().parents if (p / "utils").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import numpy as np
import pandas as pd

from utils import custom_plots as cp
from utils import custom_stats as cs
from utils.db_utils import run_query

pd.set_option("display.max_columns", 50)

## 1. Data

A discount is set per order line, not per order. One order can carry a
discounted line and a full-price line side by side. That makes the line the
right unit to compare, so this pulls from `fact_sales` (one row per line)
rather than `fact_order`; rolling up to the order grain would blend the two
groups into the same row and erase the thing being tested.

In [3]:
df = run_query("""
    SELECT
        f.quantity,
        f.discount,
        g.market
    FROM olap.fact_sales f
    JOIN olap.dim_geography g ON g.geo_key = f.geo_key
""")
df.shape

(49670, 3)

In [4]:
df["discount_group"] = np.where(df["discount"] > 0, "Discounted", "Full price")

group_counts = df["discount_group"].value_counts()
group_share = (group_counts / len(df) * 100).round(1)
print(group_counts.to_string())
print(group_share.to_string())

discount_group
Full price    28103
Discounted    21567
discount_group
Full price    56.6
Discounted    43.4


21,567 of the 49,670 lines (43.4%) carry a discount; the other 28,103 (56.6%)
sell at list price. Neither side is a rare edge case, so a real effect here,
in either direction, should show up clearly rather than hide in noise.

## 2. What quantity looks like

In [5]:
cs.summary_stats(df, cols=["quantity"]).round(3)

,column,n,n_missing,pct_missing,n_unique,mean,ci_low,ci_high,trimmed_mean,median,std,mad,iqr,cv,skew,excess_kurtosis,min,q1,q3,max
0,quantity,49670,0,0.0,14,3.471,3.451,3.491,3.158,3.0,2.275,1.483,3.0,0.655,1.363,2.284,1.0,2.0,5.0,14.0


Quantity is a small integer count: every line moves somewhere between 1 and 14
units, with only 14 distinct values across almost 50,000 rows and a median of
3. A variable this narrow and this discrete was never going to pass as normal,
and no transform turns a count into something continuous. Running Shapiro-Wilk
here would spend a paragraph confirming what the table above already shows.
The comparison below goes straight to a rank-based test because of what the
variable is, not because a normality check failed first.

In [6]:
cp.ecdf_plot(
    df, cols="quantity", group_col="discount_group",
    mark_percentiles=[0.25, 0.5, 0.75, 0.9],
    title="Quantity per Line, Discounted vs Full Price",
)

The two curves track each other almost step for step, with the discounted line
sitting a hair to the right through the middle of the range, where it takes a
slightly higher quantity to reach the same cumulative share. Both hit the
25th, 50th and 75th percentile at the same integers; the gap between them is
real but small enough that it barely shows up until you look for it.

## 3. Hypotheses

**H0**: discounted and full-price lines draw their quantity from the same
distribution; a discount changes nothing about how many units move on that
line.

**H1**: the two distributions differ.

alpha = 0.05, two-sided, tested with Mann-Whitney U. That choice was made in
section 2, before looking at the result, because quantity is a small tied
count and not because a screening test rejected normality first.

## 4. Testing the difference

In [7]:
# n_resamples=2000 for the bootstrap interval: plenty of precision at
# n ~ 25,000 per group, and it keeps this cell well under a minute.
result = cs.compare_groups(
    df, "discount_group", "quantity", test="mannwhitney",
    ci="bootstrap", n_resamples=2000, random_state=0,
)
result

,comparison,n_groups,group_sizes,test,n,statistic,p_value,decision,estimate_type,estimate,ci_low,ci_high,effect_size,effect_type,magnitude,alternative,chosen_because,flags
0,Discounted vs Full price,2,Discounted=21567; Full price=28103,mannwhitney,49670,320990669.5,1.236618e-30,reject H₀,median difference,0.0,NaN,NaN,0.059205,cliffs_delta,negligible,two-sided,user-specified,bootstrap interval undefined: the statistic ba...


The p-value is effectively zero, so H0 is rejected. But at 49,670 rows almost
any real difference clears p < 0.05, which makes the p-value the least
interesting number in that table. The one that matters is Cliff's delta: 0.06,
which lands in the "negligible" band by the usual convention. Both groups
share a median of 3 units, and that tie is exactly why the bootstrap interval
on the median gap comes back undefined — with the median pinned at 3 in both
groups, resampled medians barely move, so BCa has nothing to build an interval
from. That is a property of a heavily tied discrete variable, not a
computation error, and it is why the next cell looks at the mean instead,
where the numbers do move.

In [8]:
mean_ci = cs.bootstrap_ci(
    df, "quantity", group_col="discount_group", statistic="mean",
    n_resamples=2000,
)
mean_ci.round(4)

,group,statistic,n,estimate,ci_low,ci_high,se,method,confidence,flags
0,Discounted,mean,21567,3.5685,3.5388,3.5997,0.0157,BCa,0.95,
1,Full price,mean,28103,3.3958,3.3693,3.4231,0.0136,BCa,0.95,


Discounted lines average 3.57 units (95% CI 3.54-3.60); full-price lines
average 3.40 (3.37-3.42). The intervals do not overlap, so the gap — about
0.17 units, a 5.1% lift over the full-price mean — is a real, precisely
estimated difference. It is just a small one.

In [9]:
cp.grouped_bar_plot(
    df, group_col="discount_group", value_col="quantity", agg="mean",
    ci_method="bootstrap", n_boot=2000,
    title="Mean Quantity per Line by Discount Status",
)

The two bars are visibly separated, and the tight error bars confirm the gap
is not noise. It is also not the kind of gap that would make anyone reconsider
how they staff a warehouse: roughly one extra unit for every six discounted
lines.

## 5. Does the lift hold within every market?

A 0.17-unit lift pooled across 49,670 lines and 7 markets could be a real
per-line effect, or it could be an artefact of which markets happen to
discount more heavily and also happen to run higher quantities for other
reasons. Splitting by market and re-running the same test on each one tells
the two apart.

In [10]:
rows = []
for market, sub in df.groupby("market"):
    if sub["discount_group"].nunique() < 2:
        print(f"{market}: only one discount_group present (n={len(sub)}) "
              "- nothing to compare, excluded")
        continue
    # ci="none": the market table reports p-values and Cliff's delta, and
    # delta needs no bootstrap; skipping it keeps 6 tests fast.
    row = cs.compare_groups(sub, "discount_group", "quantity",
                             test="mannwhitney", ci="none", random_state=0)
    row.insert(0, "market", market)
    rows.append(row)

market_results = pd.concat(rows, ignore_index=True)
market_results["p_adj"] = cs.p_adjust(
    market_results["p_value"].to_numpy(dtype=float), method="holm"
)
# decision now reads off the Holm-adjusted p, not the six raw ones
market_results["decision"] = np.where(
    market_results["p_adj"] < 0.05, "reject H0", "fail to reject H0"
)

market_table = market_results[
    ["market", "group_sizes", "p_value", "p_adj", "effect_size", "magnitude",
     "decision"]
].round(4).sort_values("p_adj").reset_index(drop=True)
market_table

Canada: only one discount_group present (n=376) - nothing to compare, excluded


,market,group_sizes,p_value,p_adj,effect_size,magnitude,decision
0,EMEA,Discounted=1509; Full price=3326,0.1345,0.8072,-0.0246,negligible,fail to reject H0
1,LATAM,Discounted=4187; Full price=6042,0.1588,0.8072,-0.0161,negligible,fail to reject H0
2,Africa,Discounted=1003; Full price=3450,0.5026,1.0000,-0.0128,negligible,fail to reject H0
3,APAC,Discounted=6314; Full price=4688,0.7395,1.0000,0.0036,negligible,fail to reject H0
4,EU,Discounted=3385; Full price=5442,0.5489,1.0000,-0.0074,negligible,fail to reject H0
5,US,Discounted=5169; Full price=4779,0.3239,1.0000,-0.0112,negligible,fail to reject H0


Canada never discounts. None of its 376 lines carries one, so there is no
within-market comparison to run there. Of the six markets that do discount,
none clears significance after the Holm correction (the smallest adjusted
p-value is 0.81), and every effect size sits in "negligible" territory
(|delta| <= 0.025 throughout). More telling than the sizes is the sign: APAC
is the only market where discounted lines skew even slightly higher, matching
the pooled result; the other five (Africa, EMEA, EU, LATAM, US) point the
other way, with discounted lines running marginally *lower* quantity than full
price within that market. A pooled effect that disappears, and partly
reverses, once you control for market is a mix effect rather than a per-line
one. The aggregate 0.17-unit lift is largely explained by discounting being
more common in markets that already move more units, not by the discount
itself moving more units.

In [11]:
cp.grouped_bar_plot(
    df, group_col="market", value_col="quantity", split_col="discount_group",
    agg="mean", ci_method="bootstrap", n_boot=2000,
    title="Mean Quantity per Line by Market and Discount Status",
)

Inside every market the discounted and full-price bars sit on top of each
other within their error bars. Canada is the one market with a single bar,
because it has never run a discount, so there is nothing to compare it
against.

## 6. Does discount depth matter?

The comparison so far treats a discount as a yes/no flag. Among the lines that
are discounted, does going deeper move more product, the way the folklore
claims?

In [12]:
# 20-point bands: coarse enough to keep every bin well powered (the
# smallest, 61%+, still holds over 2,000 lines) while still tracing the
# shape of the relationship.
bins = [-0.001, 0.0, 0.20, 0.40, 0.60, 1.0]
labels = ["No discount", "1-20%", "21-40%", "41-60%", "61%+"]
df["discount_band"] = pd.cut(df["discount"].astype(float), bins=bins, labels=labels)
df["discount_band"].value_counts().reindex(labels)

discount_band
No discount    28103
1-20%          10555
21-40%          4301
41-60%          4637
61%+            2074
Name: count, dtype: int64

In [13]:
cp.grouped_bar_plot(
    df, group_col="discount_band", value_col="quantity", agg="mean",
    ci_method="bootstrap", n_boot=2000,
    title="Mean Quantity per Line by Discount Depth",
)

Mean quantity does not climb with discount depth. It peaks early and then
falls. No discount averages 3.40 units, light discounts (1-20%) average 3.74,
and from there it declines through 3.68 at 21-40% and 3.36 at 41-60% down to
2.96 at 61%+, where the median line itself drops to 2 units. The "deeper the
cut, the more it moves" story holds for the first 20 points of discount and
inverts after that.

In [14]:
discounted = df[df["discount"] > 0]
cs.correlation_test(discounted, cols=["discount", "quantity"], method="spearman")

,x,y,method,n,r,ci_low,ci_high,r_squared,p_value,p_adj,decision,magnitude,correction,n_comparisons,flags
0,discount,quantity,spearman,21567,-0.137584,-0.151039,-0.124078,0.018929,1.270943e-91,1.270943e-91,reject H₀,small,holm,1,


Restricted to lines that carry a discount, the rank correlation between
discount rate and quantity is -0.14 (95% CI -0.15 to -0.12, p < 0.001): a
small but real negative relationship, and it explains under 2% of the variance
in quantity (r-squared = 0.019). Within the discounted group, cutting the
price further is mildly associated with *fewer* units per line, not more.

## 7. What this means

Putting a discount on a line does move a little more product: 3.57 units on
average against 3.40 without one, a gap of about 0.17 units, or roughly 5%.
That gap is real, with a confidence interval that stays clear of zero. But
Cliff's delta, the number that actually measures how separated the two groups
are, comes back at 0.06, squarely "negligible," and both groups share the same
median of 3 units.

The gap does not hold up once markets are looked at one at a time. Five of the
six markets that discount at all show the effect running in the opposite
direction from the pooled result, and none of the six is significant after
correcting for running six tests. Canada does not discount at all. That
pattern says the pooled lift is mostly about which markets discount more, not
about what a discount does to a given line.

Depth does not help either. Quantity peaks at a light discount and falls from
there, down to a mean of 2.96 units — median 2 — once the discount passes 60%,
and the correlation between discount rate and quantity among discounted lines
is weakly negative.

So the first half of the store folklore does not hold in this data:
discounting nudges quantity up by a small, mostly market-driven amount, and
cutting deeper does not buy more volume. If anything it is associated with
less. Whether that small lift, or the deep-discount lines specifically, still
pays for itself is a profit question. `profit` and `discount_amount` live in
the same `fact_sales` table for whoever picks that up next; this notebook
stops at quantity on purpose.

This question does not stand alone in the project. Two other notebooks reach
the same place from different directions, and together the three are the
strongest result here. `ml/profit_regression` finds that discount depth is the
largest single driver of loss-making order lines. `ml/customer_segments` finds
that the 187 customers on the deepest discounts are a net drag on profit, and
that heavier discounting correlates with worse margin (Spearman r = -0.55) but
not with buying more often (r = -0.08). Three grains, three methods, one
conclusion: discounting at this store costs margin without buying either
volume or loyalty.